In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("ETFLIST.csv")

In [3]:

df = df.rename(columns={'Perc Hold(%)': 'Perc_Hold'})
df

,Scheme Name,Asset Type,Scheme Name.1,Company Name,Co_Name_Equity,Equity Rating,Rating,Sector,Fund Manager,Mkt Value,No Shares/Units,Perc_Hold,Price(BSE),Mkt cap,% of Equity Shares,YTM,YTC
0,360 ONE Gold ETF,Reverse Repo,360 ONE Gold ETF,CBLO,NaN,NaN,NaN,NaN,Rahul Khetawat,0.22,NaN,1.47,NaN,NaN,NaN,0.00,0
1,360 ONE Gold ETF,Gold,360 ONE Gold ETF,GOLD,NaN,NaN,NaN,NaN,Rahul Khetawat,14.35,15.0,95.89,NaN,NaN,NaN,0.00,0
2,360 ONE Silver ETF,Net CA & Others,360 ONE Silver ETF,Net CA & Others,NaN,NaN,NaN,NaN,Rahul Khetawat,0.64,NaN,5.53,NaN,NaN,NaN,0.00,0
3,360 ONE Silver ETF,Silver,360 ONE Silver ETF,Silver,NaN,NaN,NaN,NaN,Rahul Khetawat,8.56,810.0,97.11,NaN,NaN,NaN,0.00,0
4,AXIS BSE Sensex ETF,Equity,AXIS BSE Sensex ETF,Adani Ports & Special Economic Zone Ltd,Adani Ports,Average,NaN,Miscellaneous,Karthik Kumar,1.47,10148.0,1.08,"1,450.20","313,263.35",NaN,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11673,Zerodha Nifty Midcap 150 ETF,Equity,Zerodha Nifty Midcap 150 ETF,Waaree Energies Ltd,Waaree Energies,Good,NaN,Electric Equipment,Kedarnath Mirajkar,0.26,831.0,0.28,"3,138.70","90,169.58",NaN,0.00,0
11674,Zerodha Nifty Midcap 150 ETF,Equity,Zerodha Nifty Midcap 150 ETF,Yes Bank Ltd,Yes Bank,Bluechip,NaN,Banks - Private Sector,Kedarnath Mirajkar,0.98,482571.0,1.05,20.35,"63,823.04",NaN,0.00,0
11675,Zerodha Silver ETF,Reverse Repo,Zerodha Silver ETF,Clearing Corporation of India Ltd,NaN,NaN,NaN,Miscellaneous,Kedarnath Mirajkar,0.21,NaN,0.62,NaN,NaN,NaN,5.48,0
11676,Zerodha Silver ETF,Net CA & Others,Zerodha Silver ETF,Net CA & Others,NaN,NaN,NaN,NaN,Kedarnath Mirajkar,0.92,NaN,2.75,NaN,NaN,NaN,0.00,0


In [ ]:
scheme_tags = {}

# Get unique scheme names
unique_schemes = df['Scheme Name'].unique()

for scheme in unique_schemes:
    # Filter for current scheme
    scheme_data = df[df['Scheme Name'] == scheme]
    # Create pivot table: index=Asset Type, columns=Scheme Name, values=sum of Perc Hold(%)
    pivot = scheme_data.pivot_table(index='Asset Type', values='Perc_Hold', aggfunc='sum')
    pivot = pivot.rename(columns={'Perc_Hold': scheme})
    pivot_reset = pivot.reset_index()  
    # Check for Gold asset type with Perc Hold > 90
    gold_row = pivot_reset[pivot_reset['Asset Type'] == 'Gold']
    GSec_row = pivot_reset[pivot_reset['Asset Type'] == 'Govt. Securities']
    Rrepo_row = pivot_reset[pivot_reset['Asset Type'] == 'Reverse Repo']
    ForeignEquity_row = pivot_reset[pivot_reset['Asset Type'] == 'Foreign Equity']
    Silver_row = pivot_reset[pivot_reset['Asset Type'] == 'Silver']
    equity_row = pivot_reset[pivot_reset['Asset Type'] == 'Equity']
    NCD_row = pivot_reset[pivot_reset['Asset Type'] == 'NCD']
    # If AUM less than 100 Cr than tag as poor 
    # Sum 'Mkt Value' column, ignoring NaN or blank values
    # mkt_value_sum = pd.to_numeric(scheme_data['Mkt Value'], errors='coerce').sum()
    # if mkt_value_sum < 100:
    #     scheme_tags[scheme] = ['Poor', 'AUM less than 100 Cr']
    if not gold_row.empty and gold_row[scheme].iloc[0] > 90:
        scheme_tags[scheme] = ['Good', '90percent gold']
    # Add more conditions here as needed
    elif not GSec_row.empty and GSec_row[scheme].iloc[0] > 90:
        scheme_tags[scheme] = ['Good', '90percent gsec']
    elif not Rrepo_row.empty and Rrepo_row[scheme].iloc[0] > 90:
        scheme_tags[scheme] = ['Good', '90percent rev_repo']
    elif not ForeignEquity_row.empty and ForeignEquity_row[scheme].iloc[0] > 90:
        scheme_tags[scheme] = ['Average', '90percent Foreign Equity']
    elif not Silver_row.empty and Silver_row[scheme].iloc[0] > 90:
        scheme_tags[scheme] = ['notag', '90percent Silver']
    elif not equity_row.empty and equity_row[scheme].iloc[0] > 90:
        pivot = scheme_data.pivot_table(index='Equity Rating', values='Perc_Hold', aggfunc='sum')
        pivot = pivot.rename(columns={'Perc_Hold': scheme})
        pivot_reset = pivot.reset_index()

        bluechip_row = pivot_reset[pivot_reset['Equity Rating'] == 'Bluechip']
        good_row = pivot_reset[pivot_reset['Equity Rating'] == 'Good']
        poor_row = pivot_reset[pivot_reset['Equity Rating'] == 'Poor']
        if not bluechip_row.empty and bluechip_row[scheme].iloc[0] > 70:
            scheme_tags[scheme] = ['Good', '70percent bluechip in equity']
        elif not good_row.empty and (bluechip_row[scheme].iloc[0] + good_row[scheme].iloc[0]) > 75:
            scheme_tags[scheme] = ['Good', '75percent bluechip and good in equity']
        elif not good_row.empty and good_row[scheme].iloc[0] > 75:
            scheme_tags[scheme] = ['Good', '75percent bluechip and good in equity']
        elif not poor_row.empty and poor_row[scheme].iloc[0] > 40:
            scheme_tags[scheme] = ['Poor', '40percent is poor in equity']
        else:
            scheme_tags[scheme] = ['Average', 'else average in equity']

    elif not NCD_row.empty and NCD_row[scheme].iloc[0] > 90:
        pivot = scheme_data.pivot_table(index='Rating', values='Perc_Hold', aggfunc='sum')
        pivot = pivot.rename(columns={'Perc_Hold': scheme})
        pivot_reset = pivot.reset_index()

        aaa_row = pivot_reset[pivot_reset['Rating'] in ['AAA','AAA (SO)', 'AAA (CE)', 'AAA (IND)', 'Reverse repo','Cash and CE', 'Fixed Deposits', 'T Bills', 'Sovereign', 'A1+']]
        aaplus_row = pivot_reset[pivot_reset['Rating'] in ['AA+','AA+ (SO)', 'AA+ (CE)']]
        aa_row = pivot_reset[pivot_reset['Rating'] in ['AA','AA (CE)']]
        if not aaa_row.empty and aaa_row[scheme].iloc[0] > 90:
            scheme_tags[scheme] = ['Good', '90percent AAA rated bond']
        elif not aaplus_row.empty and (aaa_row[scheme].iloc[0] + aaplus_row[scheme].iloc[0]) > 95:
            scheme_tags[scheme] = ['Good', '95percent AAA and AA+ bond']
        elif not aa_row.empty and (aaa_row[scheme].iloc[0] + aaplus_row[scheme].iloc[0] + aa_row[scheme].iloc[0]) > 85:
            scheme_tags[scheme] = ['Average', '85percent AAA,AAPlus and AA Bond']
        else:
            scheme_tags[scheme] = ['poor', 'rest average in bond']

    else:
        scheme_tags[scheme] = 'OtherTag'

scheme_tags

{'360 ONE Gold ETF': ['Good', '90percent gold'],
 '360 ONE Silver ETF': ['notag', '90percent Silver'],
 'AXIS BSE Sensex ETF': ['Good', '70percent bluechip in equity'],
 'AXIS Gold ETF': ['Good', '90percent gold'],
 'AXIS Nifty 50 ETF': ['Good', '70percent bluechip in equity'],
 'AXIS Nifty AAA Bond Plus SDL Apr 2026 50:50 ETF': 'OtherTag',
 'AXIS Nifty Bank ETF': ['Good', '75percent bluechip and good in equity'],
 'AXIS Nifty Healthcare ETF': ['Good', '70percent bluechip in equity'],
 'AXIS Nifty IT ETF': ['Good', '70percent bluechip in equity'],
 'AXIS Nifty India Consumption ETF': ['Good', '70percent bluechip in equity'],
 'AXIS Nifty500 Value 50 ETF': ['Good',
  '75percent bluechip and good in equity'],
 'AXIS Silver ETF': ['notag', '90percent Silver'],
 'Aditya Birla SL BSE Sensex ETF': ['Good', '70percent bluechip in equity'],
 'Aditya Birla SL CRISIL 10 Year Gilt ETF': ['Good', '90percent gsec'],
 'Aditya Birla SL CRISIL Broad Based Gilt ETF': ['Good', '90percent gsec'],
 'Adity

In [5]:
list(scheme_tags.items())

[('360 ONE Gold ETF', ['Good', '90percent gold']),
 ('360 ONE Silver ETF', ['notag', '90percent Silver']),
 ('AXIS BSE Sensex ETF', ['Good', '70percent bluechip in equity']),
 ('AXIS Gold ETF', ['Good', '90percent gold']),
 ('AXIS Nifty 50 ETF', ['Good', '70percent bluechip in equity']),
 ('AXIS Nifty AAA Bond Plus SDL Apr 2026 50:50 ETF', 'OtherTag'),
 ('AXIS Nifty Bank ETF', ['Good', '75percent bluechip and good in equity']),
 ('AXIS Nifty Healthcare ETF', ['Good', '70percent bluechip in equity']),
 ('AXIS Nifty IT ETF', ['Good', '70percent bluechip in equity']),
 ('AXIS Nifty India Consumption ETF',
  ['Good', '70percent bluechip in equity']),
 ('AXIS Nifty500 Value 50 ETF',
  ['Good', '75percent bluechip and good in equity']),
 ('AXIS Silver ETF', ['notag', '90percent Silver']),
 ('Aditya Birla SL BSE Sensex ETF', ['Good', '70percent bluechip in equity']),
 ('Aditya Birla SL CRISIL 10 Year Gilt ETF', ['Good', '90percent gsec']),
 ('Aditya Birla SL CRISIL Broad Based Gilt ETF', ['Go

In [6]:
scheme_tags_df = pd.DataFrame(list(scheme_tags.items()), columns=['Scheme Name', 'Tag'])
scheme_tags_df

,Scheme Name,Tag
0,360 ONE Gold ETF,"[Good, 90percent gold]"
1,360 ONE Silver ETF,"[notag, 90percent Silver]"
2,AXIS BSE Sensex ETF,"[Good, 70percent bluechip in equity]"
3,AXIS Gold ETF,"[Good, 90percent gold]"
4,AXIS Nifty 50 ETF,"[Good, 70percent bluechip in equity]"
...,...,...
254,Zerodha Gold ETF,"[Good, 90percent gold]"
255,Zerodha Nifty 100 ETF,"[Good, 70percent bluechip in equity]"
256,Zerodha Nifty 1D Rate Liquid ETF,"[Good, 90percent rev_repo]"
257,Zerodha Nifty Midcap 150 ETF,"[Good, 75percent bluechip and good in equity]"


In [7]:
scheme_tags_df.to_csv("scheme_tag.csv")

In [8]:
scheme_tags_df

,Scheme Name,Tag
0,360 ONE Gold ETF,"[Good, 90percent gold]"
1,360 ONE Silver ETF,"[notag, 90percent Silver]"
2,AXIS BSE Sensex ETF,"[Good, 70percent bluechip in equity]"
3,AXIS Gold ETF,"[Good, 90percent gold]"
4,AXIS Nifty 50 ETF,"[Good, 70percent bluechip in equity]"
...,...,...
254,Zerodha Gold ETF,"[Good, 90percent gold]"
255,Zerodha Nifty 100 ETF,"[Good, 70percent bluechip in equity]"
256,Zerodha Nifty 1D Rate Liquid ETF,"[Good, 90percent rev_repo]"
257,Zerodha Nifty Midcap 150 ETF,"[Good, 75percent bluechip and good in equity]"
